In [4]:
import pandas as pd
import re
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [5]:
train_data = pd.read_csv('/content/drive/MyDrive/samsum-train.csv')
val_data = pd.read_csv('/content/drive/MyDrive/samsum-validation.csv')

In [6]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [7]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14732 entries, 0 to 14731
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        14732 non-null  object
 1   dialogue  14731 non-null  object
 2   summary   14732 non-null  object
dtypes: object(3)
memory usage: 345.4+ KB


# Data Preprocessing

In [8]:
train_data = train_data.dropna()

In [9]:
train_data.shape

(14731, 3)

In [10]:
val_data.shape

(818, 3)

In [11]:
def clean_data(text):
    text = re.sub(r'r/n/',' ',text)
    text = re.sub(r's+',' ',text)
    text = re.sub(r'<.*?>',' ',text)

    return text

In [12]:
train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary'] = train_data['summary'].apply(clean_data)

val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary'] = val_data['summary'].apply(clean_data)

# Tokenization

In [13]:
tokenizer = T5Tokenizer.from_pretrained('t5-small')

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [14]:
def tokenize(data):
    inputs = tokenizer(data['dialogue'],padding='max_length',max_length=512,truncation=True)
    target = tokenizer(data['summary'],padding='max_length',max_length=150,truncation=True)
    inputs['labels'] = target['input_ids']

    return inputs

In [15]:
train_dataset = train_data.apply(tokenize,axis=1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()

In [16]:
train_dataset[0]
# input_ids => tokenized dialogue data
# attention_mask => 1 for real data 0 for padded data
# labels => input_ids of tokenized summary data

{'input_ids': [21542, 10, 27, 13635, 7364, 3, 5, 531, 25, 241, 3, 7159, 58, 16637, 10, 10625, 55, 21542, 10, 27, 31, 195, 830, 25, 5721, 3, 10, 18, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

# Model Building

In [17]:
model = T5ForConditionalGeneration.from_pretrained('t5-small')

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [20]:
import torch

if torch.cuda.is_available():
  device = torch.device('cuda')
else:
  device = torch.device('cpu')
print('device: ', device)
model.to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [21]:
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/model',
    num_train_epochs= 4,

    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy='epoch',
    save_strategy='epoch',

    warmup_steps=500
)

In [22]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.509478,0.442736
2,0.465595,0.416693
3,0.433207,0.407599
4,0.443594,0.404876


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7368, training_loss=0.6923625130089046, metrics={'train_runtime': 2906.8737, 'train_samples_per_second': 20.271, 'train_steps_per_second': 2.535, 'total_flos': 7974880309936128.0, 'train_loss': 0.6923625130089046, 'epoch': 4.0})